# PTCG Track B RL — Kaggle Notebook (Loop K)

**Isolated training run.** Outputs → `/kaggle/working/ptcg_rl/` — register in `report/training_registry.json` before merging locally.

Settings: **GPU on**, internet on (for pip + git clone).

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path
from datetime import datetime, timezone

SLUG = os.environ.get("TRAIN_SLUG", "crustle")
TIMESTEPS = int(os.environ.get("TRAIN_TIMESTEPS", "100000"))
N_ENVS = int(os.environ.get("TRAIN_N_ENVS", "4"))
OPPONENTS = os.environ.get("TRAIN_OPPONENTS", "benchmark")
DECK = Path(os.environ.get("TRAIN_DECK", "agent_decks/pool_crustle.csv"))

WORK = Path("/kaggle/working/ptcg_rl") if Path("/kaggle/working").exists() else Path("report/kaggle_notebook_jobs") / SLUG
WORK.mkdir(parents=True, exist_ok=True)
print(SLUG, DECK, TIMESTEPS, WORK)

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gymnasium", "stable-baselines3", "sb3-contrib"])
import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
# Repo: clone to /kaggle/working/pokemon OR upload code as Kaggle dataset
REPO = Path("/kaggle/working/pokemon")
if not REPO.exists():
    raise SystemExit("Upload repo as dataset or git clone into /kaggle/working/pokemon")
sys.path.insert(0, str(REPO))
os.chdir(REPO)

In [ ]:
from rl.train_rl import train_maskable_ppo

result = train_maskable_ppo(TIMESTEPS, N_ENVS, False, DECK, OPPONENTS, 42)
src = REPO / "agent" / "models" / "rl_policy.zip"
if src.exists():
    shutil.copy2(src, WORK / f"rl_policy_{SLUG}.zip")
meta = {"loop": "K", "slug": SLUG, "deck": str(DECK), "finished": datetime.now(timezone.utc).isoformat(), **result}
(WORK / "train_result.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))